In [ ]:
## conda env vgd has openai
import base64
import os
import json
import time
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
import numpy as np
from tqdm import tqdm


In [19]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

# Preparing main index_df with new superset classes

In [20]:
## this supersets action classes file was made on 20th Oct 2026 manually by hamza.
ssv2_action_class_superset = pd.read_csv("/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/testing/ssv2_action_classes_supersets.csv")
print(ssv2_action_class_superset.shape)
ssv2_action_class_superset.sample(3)

(174, 3)


,label,template,template_superset
3,3,Bending [something] until it breaks,Bending [something]
35,35,Moving [something] across a surface without it falling down,Moving [something]
165,165,Turning the camera downwards while filming [something],Moving the camera


In [21]:
## index_df
index_df = pd.read_csv('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/index.csv')
print(index_df.shape)

index_df['cam'] = 'cam1'
index_df = index_df.rename(columns={'id':'video_id'})

print(index_df.shape)
index_df.head(2)

(155, 4)
(155, 5)


,video_id,template,placeholders,session,cam
0,260922-134214,Plugging [something] into [something],"plug, electrical extension",plug_1,cam1
1,260922-134301,Plugging [something] into [something],"plug, electrical extension",plug_1,cam1


In [22]:
# Bring in superset with a left join
index_df = pd.merge(index_df, ssv2_action_class_superset, how = 'left', on = ['template']) # LEFT JOIN
print(index_df.shape)
index_df.sample(3)

(155, 7)


,video_id,template,placeholders,session,cam,label,template_superset
2,260922-134333,Plugging [something] into [something],"plug, electrical extension",plug_1,cam1,49,Attaching [something] to [something]
53,260930-122401,Putting [something] on a surface,plug,plug_2,cam1,109,Putting [something] somewhere
154,261008-114959,Moving [something] and [something] closer to each other,"pen, board marker",moveCloserToEachOther_1,cam1,37,Moving [something] and [something]


In [23]:
print(f"Total classes:             {ssv2_action_class_superset.template.nunique()}")
print(f"Total superset classes:    {ssv2_action_class_superset.template_superset.nunique()}")
print("-"*30)
print(f"myvideos total classes: {index_df.template.nunique()}")
print(f"myvideos superset classes: {index_df.template_superset.nunique()}")

Total classes:             174
Total superset classes:    38
------------------------------
myvideos total classes: 15
myvideos superset classes: 9


In [24]:
index_df.template_superset.value_counts()

template_superset
Putting [something] somewhere                          51
Moving [something]                                     34
Pretending to do something without touching objects    21
Attaching [something] to [something]                   16
Stacking things                                         8
Showing or exhibiting [something]                       8
Bending [something]                                     6
Moving [something] and [something]                      6
Tearing [something]                                     5
Name: count, dtype: int64

### select videos

In [26]:
## special session
index_df['session_special'] = np.where(index_df.session.isin(['plug_2','magnet_2','lady_2','books_2']), 'distractors_in_video',
                               np.where(index_df.session.isin(['plug_1','magnet_1','lady_1','books_1']), 'session_1_control', 
                                        np.where(index_df.session.isin(['plugging_1','showing_photo_of_1','attaching_1','piling_1']), 'session2_specialActions_commonobjects',
                                                 np.where(index_df['session'].str.startswith('mime'), 'miming_videos', 

                                                          'mime_actions_control_videos'))))

index_df.tail(2)

,video_id,template,placeholders,session,cam,label,template_superset,session_special
153,261008-114942,Moving [something] and [something] closer to each other,"wallet, pen",moveCloserToEachOther_1,cam1,37,Moving [something] and [something],mime_actions_control_videos
154,261008-114959,Moving [something] and [something] closer to each other,"pen, board marker",moveCloserToEachOther_1,cam1,37,Moving [something] and [something],mime_actions_control_videos


In [27]:
index_df[index_df.session_special.isin(['distractors_in_video','session_1_control','session2_specialActions_commonobjects'])].groupby(['session_special']).agg(
    vids=('video_id','count'),
    nunique_superset_classes=('template_superset', lambda x: x.nunique()),
    unique_superset_classes=('template_superset', lambda x: ','.join(map(str, x.unique())))
)

,vids,nunique_superset_classes,unique_superset_classes
session_special,,,
distractors_in_video,80,2,"Putting [something] somewhere,Moving [something]"
session2_specialActions_commonobjects,12,3,"Attaching [something] to [something],Showing or exhibiting [something],Stacking things"
session_1_control,20,3,"Attaching [something] to [something],Stacking things,Showing or exhibiting [something]"


In [28]:
## i am going to pick the videos that were classified correctly by the vjepa2-b-attnprobe that i trained on full ssv2 dataset.
## notebook: vgd/experiment_vjepa_attn_probe_trainings/results/inference_results/myvideos_1_2_3_100pc_model/analysis_myvideos_100.ipynb


In [57]:
vids_list_normal = [
    '260922-134333', # session_1_control | plug, electrical extension --> Plugging [something] into [something] 
    '260922-134419', # session_1_control | plug, electrical extension --> Plugging [something] into [something] 
    '260922-134433', # session_1_control | plug, electrical extension --> Plugging [something] into [something] 
    '260922-143400', # session_1_control | magnet, tripod --> Attaching [something] to [something] 
    '260922-144244', # session_1_control | books --> Piling [something] up 
    '260922-150740', # session_1_control | lady --> Showing a photo of [something] to the camera 
    '260922-151251', # session_1_control | paper --> Tearing [something] into two pieces 
    '260930-144612', # session2_specialActions_commonobjects | cable, phone --> Plugging [something] into [something] 
    '260930-144637', # session2_specialActions_commonobjects | cable, ear phone case --> Plugging [something] into [something] 
    '260930-145329', # session2_specialActions_commonobjects | board marker cap, board marker --> Attaching [something] to [something]
]

In [58]:
vids_list_distractors = [
    '260922-141236', # distractors_in_video | water bottle, board marker --> Putting [something] next to [something] 
    '260922-141249', # distractors_in_video | cup --> Putting [something] on a surface 
    '260930-122317', # distractors_in_video | plug, electrical extension --> Putting [something] next to [something] 
    '260930-122340', # distractors_in_video | plug, water bottle --> Putting [something] next to [something] 
    '260930-122401', # distractors_in_video | plug --> Putting [something] on a surface 

    '260930-122827', # --> distractors_in_video | plug, electrical extension --> Putting [something] onto [something] 

    '260930-141745', # distractors_in_video | magnet, water bottle --> Putting [something] next to [something] 
    '260930-141829', # distractors_in_video | pen, magnet --> Putting [something] next to [something] 
    '260930-143410', # distractors_in_video | wallet, books --> Putting [something] next to [something] 
    '260930-143429', # distractors_in_video | books --> Putting [something] on a surface 
    '260930-143444', # distractors_in_video | books --> Putting [something] on a surface 
]

In [62]:
import shutil
# ------------- chunk to copy videos ------------- #
source_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/"
dest_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/zz_temp_dump/"

# 2. Clear destination directory if it exists, then recreate it
if os.path.exists(dest_dir):
    shutil.rmtree(dest_dir)
os.makedirs(dest_dir, exist_ok=True)

# 3. Get unique video IDs and copy matching files
unique_video_ids = vids_list_normal + vids_list_distractors

for filename in os.listdir(source_dir):
    if any(video_id in filename for video_id in unique_video_ids):
        shutil.copy(os.path.join(source_dir, filename), os.path.join(dest_dir, filename))
# ------------------------------------------------ #

# VLM Evaluation

In [31]:
"""
from openai import OpenAI
API_KEY = 'SECRET'
API_ENDPOINT = 'https://chat-ai.academiccloud.de/v1'
client = OpenAI(api_key=API_KEY, base_url=API_ENDPOINT)
client.models.list().data

import base64


with open("clip.mp4", "rb") as f:
    b64 = base64.b64encode(f.read()).decode()

resp = client.chat.completions.create(
model="glm-5.3-flash",
messages=[{
"role": "user",
"content": [
{"type": "text", "text": "Describe what the animal is doing in this video."},
{"type": "video_url", "video_url": {"url": f"data:video/mp4;base64,{b64}"}},
],
}],
)
print(resp.choices[0].message.content)

"""

'\nfrom openai import OpenAI\nAPI_KEY = \'SECRET\'\nAPI_ENDPOINT = \'https://chat-ai.academiccloud.de/v1\'\nclient = OpenAI(api_key=API_KEY, base_url=API_ENDPOINT)\nclient.models.list().data\n\nimport base64\n\n\nwith open("clip.mp4", "rb") as f:\n    b64 = base64.b64encode(f.read()).decode()\n\nresp = client.chat.completions.create(\nmodel="glm-5.3-flash",\nmessages=[{\n"role": "user",\n"content": [\n{"type": "text", "text": "Describe what the animal is doing in this video."},\n{"type": "video_url", "video_url": {"url": f"data:video/mp4;base64,{b64}"}},\n],\n}],\n)\nprint(resp.choices[0].message.content)\n\n'

In [32]:
# Load environment variables from .env
load_dotenv()

client = OpenAI(
    api_key=os.getenv("GWDG_OPENAI_API_KEY"), 
    base_url="https://chat-ai.academiccloud.de/v1"
    )
client

In [47]:
labels_list = index_df.template_superset.unique().tolist()
labels_list.sort()

# id2label = dict()
# for i, v in enumerate(a):
#     id2label[i] = v

print(labels_list)

['Attaching [something] to [something]', 'Bending [something]', 'Moving [something]', 'Moving [something] and [something]', 'Pretending to do something without touching objects', 'Putting [something] somewhere', 'Showing or exhibiting [something]', 'Stacking things', 'Tearing [something]']


In [ ]:
# Encode video to base64
video_path = '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw//260922-134333_cam1.avi'

with open(video_path, "rb") as f:
    b64_video = base64.b64encode(f.read()).decode("utf-8")

In [48]:
system_prompt = (
    "You are an expert video action classification model. "
    "Analyze video clips carefully and classify the primary action into a closed set of categories."
)

user_prompt = f"""
Analyze this short video and classify the action performed.

Candidate Action Classes:
{json.dumps(labels_list, indent=2)}

Instructions:
1. Assign a probability (float between 0.0 and 1.0) to EVERY class. The probabilities MUST sum to 1.0.
2. Select the top predicted class name.
3. Provide a brief 1-sentence reasoning for your visual observation.

Output strictly in JSON matching this schema:
{{
  "reasoning": "<1-sentence visual description>",
  "predicted_class": "<exact class name from list>",
  "probabilities": {{
    "{labels_list[0]}": 0.0,
    "{labels_list[1]}": 0.0,
    ...
  }}
}}
"""
# print(user_prompt)

In [49]:
start_time = time.time()

response = client.chat.completions.create(
    model="glm-5.3-flash",
    messages=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {"type": "text", "text": user_prompt},
                {
                    "type": "video_url",
                    "video_url": {"url": f"data:video/avi;base64,{b64_video}"},
                },
            ],
        },
    ],
    response_format={"type": "json_object"},
    temperature=0.1,  # Low temperature for deterministic evaluation outputs
)

elapsed_time = round(time.time() - start_time, 2)

In [54]:
# -----------------------------------------------------------------------------
# 4. Parse Output & Save Metadata
# -----------------------------------------------------------------------------
# Extract JSON prediction content
prediction_json = json.loads(response.choices[0].message.content)

# Extract token & performance metadata
usage = response.usage

run_result = {
    "video_file": os.path.basename(video_path),
    "model": response.model,
    "elapsed_seconds": elapsed_time,
    "prompt_tokens": usage.prompt_tokens if usage else None,
    "completion_tokens": usage.completion_tokens if usage else None,
    "total_tokens": usage.total_tokens if usage else None,
    "predicted_class": prediction_json.get("predicted_class"),
    "reasoning": prediction_json.get("reasoning"),
    "probabilities": prediction_json.get("probabilities", {}),
}

# Append result as a single JSON line
with open('data/vlm_eval_results/action_predictions.jsonl', "a", encoding="utf-8") as f:
    f.write(json.dumps(run_result) + "\n")

print(json.dumps(run_result, indent=2))

{
  "video_file": "260922-134333_cam1.avi",
  "model": "glm-5.3-flash",
  "elapsed_seconds": 7.53,
  "prompt_tokens": 5074,
  "completion_tokens": 332,
  "total_tokens": 5406,
  "predicted_class": "Attaching [something] to [something]",
  "reasoning": "A person holds a power strip with one hand and inserts a black plug into one of its sockets, attaching the plug to the power strip.",
  "probabilities": {
    "Attaching [something] to [something]": 0.85,
    "Bending [something]": 0.0,
    "Moving [something]": 0.05,
    "Moving [something] and [something]": 0.03,
    "Pretending to do something without touching objects": 0.0,
    "Putting [something] somewhere": 0.05,
    "Showing or exhibiting [something]": 0.0,
    "Stacking things": 0.0,
    "Tearing [something]": 0.0
  }
}


## For loop over many videos

In [ ]:
# Load environment variables from .env
load_dotenv()

client = OpenAI(
    api_key=os.getenv("GWDG_OPENAI_API_KEY"), 
    base_url="https://chat-ai.academiccloud.de/v1"
    )

In [70]:
## Define these before for loop once

labels_list = index_df.template_superset.unique().tolist()
labels_list.sort()

system_prompt = (
    "You are an expert video action classification model. "
    "Analyze video clips carefully and classify the primary action into a closed set of categories."
)

user_prompt = f"""
Analyze this short video and classify the action performed.

Candidate Action Classes:
{json.dumps(labels_list, indent=2)}

Instructions:
1. Assign a probability (float between 0.0 and 1.0) to EVERY class. The probabilities MUST sum to 1.0.
2. Select the top predicted class name.
3. Provide a brief 1-sentence reasoning for your visual observation.

Output strictly in JSON matching this schema:
{{
  "reasoning": "<1-sentence visual description>",
  "predicted_class": "<exact class name from list>",
  "probabilities": {{
    "{labels_list[0]}": 0.0,
    "{labels_list[1]}": 0.0,
    ...
  }}
}}
"""

videos_dir = '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw/'
video_paths = [videos_dir + video + '_cam1.avi' for video in vids_list_normal[1:]]
video_paths

['/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw/260922-134419_cam1.avi',
 '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw/260922-134433_cam1.avi',
 '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw/260922-143400_cam1.avi',
 '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw/260922-144244_cam1.avi',
 '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw/260922-150740_cam1.avi',
 '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw/260922-151251_cam1.avi',
 '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw/260930-144612_cam1.avi',
 '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw/260930-144637_cam1.avi',
 '/user/h.siddiqui/u27673/.project/dir.p

In [ ]:
from tqdm import tqdm

pbar = tqdm(video_paths, desc="Processing Videos")

for video_path in pbar:
    # Load video in b64
    with open(video_path, "rb") as f:
        b64_video = base64.b64encode(f.read()).decode("utf-8")

    # API call
    start_time = time.time()

    response = client.chat.completions.create(
        model="glm-5.3-flash",
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": user_prompt},
                    {
                        "type": "video_url",
                        "video_url": {"url": f"data:video/avi;base64,{b64_video}"},
                    },
                ],
            },
        ],
        response_format={"type": "json_object"},
        temperature=0.1,  # Low temperature for deterministic evaluation outputs
    )

    elapsed_time = round(time.time() - start_time, 2)

    # Parse Output & Save data
    prediction_json = json.loads(response.choices[0].message.content)
    usage = response.usage

    run_result = {
        "video_file": os.path.basename(video_path),
        "model": response.model,
        "elapsed_seconds": elapsed_time,
        "prompt_tokens": usage.prompt_tokens if usage else None,
        "completion_tokens": usage.completion_tokens if usage else None,
        "total_tokens": usage.total_tokens if usage else None,
        "predicted_class": prediction_json.get("predicted_class"),
        "reasoning": prediction_json.get("reasoning"),
        "probabilities": prediction_json.get("probabilities", {}),
    }

    with open('data/vlm_eval_results/action_predictions_normal.jsonl', "a", encoding="utf-8") as f:
        f.write(json.dumps(run_result) + "\n")

    # print(json.dumps(run_result, indent=2))

Processing Videos: 100%|██████████| 9/9 [11:12<00:00, 74.71s/it] 


In [72]:
videos_dir = '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw/'
video_paths = [videos_dir + video + '_cam1.avi' for video in vids_list_distractors]
video_paths

['/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw/260922-141236_cam1.avi',
 '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw/260922-141249_cam1.avi',
 '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw/260930-122317_cam1.avi',
 '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw/260930-122340_cam1.avi',
 '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw/260930-122401_cam1.avi',
 '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw/260930-122827_cam1.avi',
 '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw/260930-141745_cam1.avi',
 '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw/260930-141829_cam1.avi',
 '/user/h.siddiqui/u27673/.project/dir.p

In [73]:
from tqdm import tqdm

pbar = tqdm(video_paths, desc="Processing Videos")

for video_path in pbar:
    # Load video in b64
    with open(video_path, "rb") as f:
        b64_video = base64.b64encode(f.read()).decode("utf-8")

    # API call
    start_time = time.time()

    response = client.chat.completions.create(
        model="glm-5.3-flash",
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": user_prompt},
                    {
                        "type": "video_url",
                        "video_url": {"url": f"data:video/avi;base64,{b64_video}"},
                    },
                ],
            },
        ],
        response_format={"type": "json_object"},
        temperature=0.1,  # Low temperature for deterministic evaluation outputs
    )

    elapsed_time = round(time.time() - start_time, 2)

    # Parse Output & Save data
    prediction_json = json.loads(response.choices[0].message.content)
    usage = response.usage

    run_result = {
        "video_file": os.path.basename(video_path),
        "model": response.model,
        "elapsed_seconds": elapsed_time,
        "prompt_tokens": usage.prompt_tokens if usage else None,
        "completion_tokens": usage.completion_tokens if usage else None,
        "total_tokens": usage.total_tokens if usage else None,
        "predicted_class": prediction_json.get("predicted_class"),
        "reasoning": prediction_json.get("reasoning"),
        "probabilities": prediction_json.get("probabilities", {}),
    }

    with open('data/vlm_eval_results/action_predictions_distractors.jsonl', "a", encoding="utf-8") as f:
        f.write(json.dumps(run_result) + "\n")

    # print(json.dumps(run_result, indent=2))

Processing Videos:   0%|          | 0/11 [05:44<?, ?it/s]


KeyboardInterrupt: 

# Loading and analyzing the jsonl outputs of the VLM

In [56]:
# 1. Load jsonl file directly into a DataFrame
df = pd.read_json("data/vlm_eval_results/action_predictions.jsonl", lines=True)

# 2. Expand the nested 'probabilities' dictionary into clean individual columns
probs_df = pd.json_normalize(df["probabilities"])

# 3. Merge metadata with probability columns
final_df = pd.concat([df.drop(columns=["probabilities"]), probs_df], axis=1)

print(final_df.head())

               video_file          model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens                       predicted_class                                                                                                                            reasoning  Attaching [something] to [something]  Bending [something]  Moving [something]  Moving [something] and [something]  Pretending to do something without touching objects  Putting [something] somewhere  Showing or exhibiting [something]  Stacking things  Tearing [something]
0  260922-134333_cam1.avi  glm-5.3-flash             7.53           5074                332          5406  Attaching [something] to [something]  A person holds a power strip with one hand and inserts a black plug into one of its sockets, attaching the plug to the power strip.                                  0.85                  0.0                0.05                                0.03                                                  0.0       